In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set figure background to white, edgecolor to black, and font size
plt.rcParams.update({
    'figure.facecolor': 'white',
    'figure.edgecolor': 'black',
    # 'figure.dpi': 100,
    'axes.facecolor': 'white',
    'font.size': 18
})

models = {
    'tinies': [
        'bnn_126-28-2',
        'distilled_126-28-2',
        'full_126-28-2',
    ],
    'denses': [
        'bnn_126-42-2',
        'distilled_126-42-2',
        'full_126-42-2',
    ],
    'wides': [
        'bnn_128-32-8-2',
        'distilled_128-32-8-2',
        'full_128-32-16-2',
    ],
}

datasets = [
    'CICIDS2017',
    'UNSW-NB15',
    'UNSW-NB15-custom'
]

In [10]:
datasets = ['CICIDS2017', 'UNSW-NB15', 'UNSW-NB15-custom']
metrics  = ['accuracy', 'precision', 'recall', 'f1']
labels   = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

def load_results(model_name: str, dataset: str) -> pd.DataFrame:
    """
    Read /home/sgeraci/slu/inet-hynn/pysrc/results/<model>/results_<model>_<dataset>.csv
    with columns: metric, mean, std. Return indexed by metric with renamed columns.
    """
    path = Path(f"/home/sgeraci/slu/inet-hynn/pysrc/results/{model_name}/"
                f"results_{model_name}_{dataset}.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")
    df = pd.read_csv(path).set_index('metric')
    return df.rename(columns={'mean': f'{model_name}_mean',
                              'std':  f'{model_name}_std'})

for category, model_list in models.items():
    for dataset in datasets:

        print(f"Category: {category}")
        print(f"Models:   {model_list}")
        print(f"Dataset:  {dataset}\n")

        # Load and join results for the current dataset
        dfs = [load_results(m, dataset) for m in model_list]
        df_all = pd.concat(dfs, axis=1)

        n_metrics = len(metrics)
        n_models  = len(model_list)

        x = np.arange(n_metrics)
        group_width = 0.7
        bar_w = group_width / n_models

        fig, ax = plt.subplots(figsize=(8, 5))

        hatches = ['o', '//', 'o//']  # length == n_models

        for i, (model, h) in enumerate(zip(model_list, hatches)):
            means = [df_all.loc[m, f'{model}_mean'] for m in metrics]
            ax.bar(
                x + (i - (n_models - 1) / 2) * (bar_w + 0.02),
                means,
                width=bar_w,
                capsize=5,
                label=model.split('_')[0].upper(),
                hatch=h
            )

        # styling
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(0.7)

        ax.margins(x=0.1, y=0.1)
        ax.set_ylim(0.4, 1.09)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel('Score')
        ax.legend(loc='upper left', ncol=n_models, fontsize='small')

        plt.tight_layout()

        # ensure output dir exists, save first, then (optionally) show
        out_dir = Path(f"results/comparison_{category}")
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"model_comparison-{dataset}.png"
        fig.savefig(out_path, edgecolor='black', dpi=300)
        print(f"Saved plot as {out_path}\n")
        # plt.show()  # uncomment if you want interactive display
        plt.close(fig)


Category: tinies
Models:   ['bnn_126-28-2', 'distilled_126-28-2', 'full_126-28-2']
Dataset:  CICIDS2017

Saved plot as results/comparison_tinies/model_comparison-CICIDS2017.png

Category: tinies
Models:   ['bnn_126-28-2', 'distilled_126-28-2', 'full_126-28-2']
Dataset:  UNSW-NB15

Saved plot as results/comparison_tinies/model_comparison-UNSW-NB15.png

Category: tinies
Models:   ['bnn_126-28-2', 'distilled_126-28-2', 'full_126-28-2']
Dataset:  UNSW-NB15-custom

Saved plot as results/comparison_tinies/model_comparison-UNSW-NB15-custom.png

Category: denses
Models:   ['bnn_126-42-2', 'distilled_126-42-2', 'full_126-42-2']
Dataset:  CICIDS2017

Saved plot as results/comparison_denses/model_comparison-CICIDS2017.png

Category: denses
Models:   ['bnn_126-42-2', 'distilled_126-42-2', 'full_126-42-2']
Dataset:  UNSW-NB15

Saved plot as results/comparison_denses/model_comparison-UNSW-NB15.png

Category: denses
Models:   ['bnn_126-42-2', 'distilled_126-42-2', 'full_126-42-2']
Dataset:  UNSW-NB15